# 07 · SASRec (ticket #16)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `candidates/sasrec_{val,test}_run0.parquet`
- `sasrec/sasrec_run0.pt`: checkpoint (trọng số, tham số, danh mục bài, lịch sử huấn luyện)
- `sasrec/sasrec_run0.json`: tham số, lịch sử từng epoch (loss, NDCG@10 trên 2.000 phiên val), độ chính xác trên toàn bộ val

SASRec chuyển từ `pmixer/SASRec.pytorch` (Apache-2.0, ADR 0002); nguồn và thay đổi ở `src/models/sasrec/NOTICE.md`. Tham số cố định: hidden 128, 2 khối, 2 head, dropout 0,2, lr 0,001, batch 256, maxlen 50, cross-entropy với 256 mẫu âm dùng chung. Dừng sớm khi NDCG@10 trên mẫu val không tăng 10 epoch, hoặc sau 40 phút huấn luyện. Chuỗi đệm bên phải (seed ở vị trí 1–5), khác bản gốc; lý do ở mục Kết quả, lần 1.

Settings: **Accelerator: GPU** (T4 hoặc P100), bật **Internet** để clone repo. torch có sẵn trên Kaggle. Ước tính dưới 1 giờ.

Repo được clone vào `/tmp` (không phải `/kaggle/working`), nên output chỉ có thư mục `out/`.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #16 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU" 

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.sasrec.cli import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # huấn luyện, dừng sớm theo val, xuất candidates + checkpoint
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lịch sử huấn luyện và NDCG@10 trên toàn bộ val, so với baseline (#9), ALS (#14), item2vec (#15)
import pandas as pd

REFERENCE_NDCG = {"popularity": 0.0040, "itemknn": 0.1297, "als": 0.0967, "item2vec": 0.0672}
print(pd.DataFrame(report["history"]).round(4).to_string(index=False))
ndcg = report["val"]["ndcg"]
print(f"\nSASRec, {len(report['history'])} epoch | NDCG@10 val {ndcg:.4f} | HitRate@10 {report['val']['hit_rate']:.4f} | MRR {report['val']['mrr']:.4f}")
for name, ref in REFERENCE_NDCG.items():
    print(f"  so với {name} {ref}: {ndcg / ref - 1:+.1%}")

## Đăng output

1. **Save Version → Save & Run All (Commit)**, giữ Accelerator GPU.
2. Tab Output của version vừa chạy → **New Dataset** tên `mixnext-sasrec`, **Visibility: Private** (dữ liệu sinh từ MPD), License bất kỳ. Dataset chứa checkpoint (tiêu chí #16) và candidates.
   - Notebook sau Add Input bộ này và tìm file bằng `Path("/kaggle/input").rglob(<tên file>)`; nạp lại mô hình bằng `SASRec.load(<đường dẫn .pt>, sessions)`.

## Kết quả trên val

### Lần 1 (Kaggle, Tesla T4, torch 2.11.0+cu130, 26/11/2026, 305 giây): đệm trái như bản gốc, patience 3

| epoch | loss | giây | NDCG@10 (2.000 phiên val) |
|---:|---:|---:|---:|
| 1 | 4,5424 | 11,6 | 0,0047 |
| 2 | 4,2534 | 20,6 | 0,0038 |
| 3 | 3,9985 | 29,5 | 0,0062 |
| 4 | 3,7735 | 38,7 | 0,0076 |
| 5 | 3,5674 | 47,6 | 0,0126 |
| 6 | 3,3766 | 56,7 | 0,0138 |
| 7 | 3,1480 | 65,9 | 0,0179 |
| 8 | 3,0040 | 75,0 | 0,0183 |
| 9 | 2,8867 | 84,2 | 0,0203 |
| 10 | 2,7700 | 93,4 | 0,0191 |
| 11 | 2,6938 | 102,5 | 0,0207 |
| 12 | 2,5934 | 111,7 | 0,0249 |
| 13 | 2,4915 | 120,7 | 0,0301 |
| 14 | 2,4299 | 130,0 | 0,0307 |
| 15 | 2,3835 | 139,0 | 0,0299 |
| 16 | 2,3284 | 148,1 | 0,0301 |
| 17 | 2,2944 | 157,2 | 0,0347 |
| 18 | 2,2411 | 166,3 | 0,0391 |
| 19 | 2,1980 | 175,5 | 0,0400 |
| 20 | 2,1868 | 184,6 | 0,0417 |
| 21 | 2,1589 | 193,7 | 0,0464 |
| 22 | 2,1116 | 203,0 | 0,0445 |
| 23 | 2,0797 | 212,0 | 0,0469 |
| 24 | 2,0767 | 221,1 | 0,0472 |
| 25 | 2,0433 | 230,3 | 0,0508 |
| 26 | 2,0335 | 239,3 | 0,0505 |
| 27 | 2,0110 | 248,5 | 0,0506 |
| 28 | 1,9918 | 257,6 | 0,0507 |

Dừng ở epoch 28 (4,3 phút huấn luyện). NDCG@10 trên toàn bộ val **0,0483**, HitRate@10 0,0959, MRR 0,0424: kém item2vec (0,0672), ALS (0,0967) và item-kNN liền kề (0,1297).

Hai vấn đề trong cách cài, sửa ở lần 2:
- **Dừng quá sớm:** NDCG@10 trên mẫu val vẫn đang lên (0,042 ở epoch 20 → 0,051 ở epoch 25) và loss vẫn giảm đều, nhưng epoch 26–28 dao động quanh 0,0505–0,0507 dưới mốc 0,0508 nên patience 3 dừng lại. Dao động cỡ này là bình thường trên 2.000 phiên → patience 10, vẫn giới hạn 40 phút.
- **Lệch ngữ cảnh lúc gợi ý:** đệm trái đặt 5 bài seed ở vị trí 46–50 sau 45 ô đệm. Khi học, một bài ở vị trí 46–50 hầu như luôn có khoảng 45 bài thật phía trước (playlist MPD thường dài hơn 50 bài), nên mô hình gần như chưa gặp ngữ cảnh "5 bài sau 45 ô đệm". → Đệm phải: chuỗi luôn bắt đầu ở vị trí 1, seed ở vị trí 1–5, giống hệt "5 bài đầu một cửa sổ → bài thứ 6" lúc học (`src/models/sasrec/NOTICE.md`).

### Lần 2 (Kaggle, Tesla T4, torch 2.11.0+cu130, 26/11/2026, 1.340 giây): đệm phải, patience 10

| epoch | loss | giây | NDCG@10 (2.000 phiên val) |
|---:|---:|---:|---:|
| 1 | 4,4734 | 12,6 | 0,0070 |
| 10 | 2,7067 | 95,6 | 0,0217 |
| 20 | 2,1528 | 189,8 | 0,0430 |
| 30 | 1,9352 | 283,3 | 0,0504 |
| 40 | 1,8047 | 376,6 | 0,0582 |
| 50 | 1,7130 | 470,1 | 0,0629 |
| 60 | 1,6507 | 563,3 | 0,0703 |
| 70 | 1,6025 | 656,6 | 0,0712 |
| 80 | 1,5625 | 750,1 | 0,0722 |
| 90 | 1,5205 | 843,3 | 0,0722 |
| 100 | 1,4893 | 936,6 | 0,0745 |
| 110 | 1,4673 | 1029,9 | 0,0787 |
| 120 | 1,4429 | 1123,9 | 0,0776 |
| 128 | 1,4300 | 1199,0 | **0,0838** |
| 130 | 1,4154 | 1217,6 | 0,0818 |
| 138 | 1,4110 | 1293,1 | 0,0832 |

(Lịch sử đủ 138 epoch trong `sasrec/sasrec_run0.json` của Dataset `mixnext-sasrec`.)

**Kết quả:** học 138 epoch (21,5 phút huấn luyện, tổng 22,3 phút gồm xuất candidates), dừng vì 10 epoch không vượt mốc của epoch 128; giữ trạng thái epoch 128. NDCG@10 trên toàn bộ val **0,0809**, HitRate@10 0,1510, MRR 0,0695.

| mô hình gu | HitRate@10 | NDCG@10 | MRR |
|---|---:|---:|---:|
| popularity | 0,0089 | 0,0040 | 0,0037 |
| item2vec | 0,1242 | 0,0672 | 0,0567 |
| SASRec (lần 2) | 0,1510 | 0,0809 | 0,0695 |
| ALS | 0,1709 | 0,0967 | 0,0829 |
| item-kNN liền kề | 0,1921 | **0,1297** | 0,1165 |

- Sửa đệm và patience tăng NDCG@10 từ 0,0483 lên 0,0809 (+67%).
- SASRec hơn item2vec 20% nhưng kém ALS 16% và item-kNN liền kề 38%. Thứ tự bài có ích (SASRec học được bài kế tiếp, test trên vòng đồ chơi), nhưng với seed chỉ 5 bài, quan hệ trực tiếp "bài cuối → bài kế" của item-kNN liền kề vẫn mạnh nhất.
- NDCG@10 trên mẫu val còn nhích lên chậm và dao động khoảng ±0,003 giữa các epoch; học lâu hơn hoặc mô hình lớn hơn có thể tăng thêm chút ít nhưng nằm ngoài ngân sách 1 giờ của #16.

NDCG@10 ở đây tính trên toàn bộ val, trong đó có 2.000 phiên dùng để dừng sớm, nên hơi lạc quan; ALS và item2vec cũng chọn tham số trên val nên so sánh vẫn công bằng. Con số không chọn trên val là bảng test ở #26.